In [5]:
import requests
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from sqlalchemy import create_engine
import time
import cloudscraper
import re

In [6]:
list_am_main = 'https://www.list.am'
list_am_url = f"{list_am_main}/category/1472?n=0&bid=49&mid=957%2C3284&_a2_1=2012&_a2_2=2017&_a27=0&_a22=0&_a102=0"

In [38]:
def get_html_response(url):
    scraper = cloudscraper.create_scraper()
    response = scraper.get(url)
    response_text = response.text
    soup = BeautifulSoup(response_text, "html.parser")
    return soup


def get_cars(soup):
    search_results = soup.find('div' ,attrs={'class':'gl'})
    return search_results

def get_hrefs(cars):
    hrefs = [list_am_main + link['href'] for link in cars.find_all('a',attrs={'class':"class"})]
    return hrefs

def get_specific_car_content(href):
    scraper = cloudscraper.create_scraper()
    html = scraper.get(href).text
    #html = requests.get(href).text
    soup = BeautifulSoup(html, "html.parser")
    return soup

def get_car_name(car_soup):
    return car_soup.find('title').text.split(' - ')[0]

def get_car_vin(car_soup):
    try:
        return car_soup.find('div',attrs={'class':"pad-left-6"}).text.strip()
    except:
        return

def get_car_year(car_soup):
    return car_soup.find('a',attrs={'class':"grey-text"}).text

def get_car_metadata(car_soup):
    data_dict = {}
    all_details = car_soup.find('div', class_='vi')
    all_details

    car_detils = all_details.find_all('div',class_='attr g new')
    for detail in car_detils:
        sub_details = detail.find_all('div',class_='at2')
        for sub_detail in sub_details:
            names = sub_detail.find_all('div')
            for name in names[1:]:
                titles = name.find_all('p')
                if len(titles) > 1:
                    tag_element,name_element = titles
                else:
                    tag_element =  titles[0]
                    name_element = titles[0]
                data_dict[name_element.text] = tag_element.text
                    
    return data_dict


def get_add_info(car_soup):
    add_infos = car_soup.find_all('div',attrs={'class':"nottii bltitle medium"})
    count = 0
    add_exists = False
    for i in add_infos:
        if i.text == 'Լրացուցիչ':
            add_exists = True
            break
        else:
            count += 1
    if add_exists:
        add_info = car_soup.find_all('div',attrs={'class':"ad-options"})[count].text.strip()
    else:
        add_info = None
    return add_info

def get_car_seller_phone(car_soup):
    try:
        return car_soup.find('a' , attrs={'id':"callBtnOptional1"}).text.strip()
    except:
        return car_soup.find('a' , attrs={'id':"callBtn1"}).text.strip()

def get_seller_id(car_soup):
    return car_soup.find('a',class_ = 'n')['href'].split('/')[-1]

def get_car_price(car_soup):
    
    while True:
        element = car_soup.find('span', class_='price x')
        if element:
            break
        else:
            time.sleep(2)
            print('Trying find price')

    return element.text.strip()

def get_add_info(car_soup):
    def extract_post_id(span):
        return span.text.split(' ')[-1]
        return re.search(r'\d+', span.text).group()

    def extract_create_date(span):
        return span.text.split(' ')[-1]

    def extract_update_date(span):
        if span: 
            return ' '.join(span.text.split(' ')[-2:]) 
    
    description = car_soup.find('div', class_='vi').find('div', class_='body').text
    other_info = car_soup.find('div', class_='vi').find('div', class_='footer').find_all('span')
    
    if len(other_info) == 3:
        post_id_span,create_date_span,update_date_span = other_info
    else:
        post_id_span,create_date_span = other_info
        update_date_span = None
    post_id,create_date,update_date = extract_post_id(post_id_span) ,extract_create_date(create_date_span),extract_update_date(update_date_span)
    return description,post_id,create_date,update_date

In [8]:
soup = get_html_response(list_am_url)
cars = get_cars(soup)
hrefs = get_hrefs(cars)

In [9]:
print(f'Found {len(hrefs)} cars')

Found 27 cars


In [28]:
a = {'a':5}
b = {'b':7}

a.update(b)

In [29]:
a

{'a': 5, 'b': 7}

In [48]:
def reverse_keys(metadata,new_keys):
    new_dict = {}
    for i in metadata.items():
        for j in new_keys:
            if j in i:
                new_dict[j] = i[0]
    
    for value in new_dict.values():
        del metadata[value]
    metadata.update(new_dict)
    return metadata


['https://www.list.am/item/23006282?ld_src=2']

In [51]:
count = 0
cars_list = []
for href in filter(lambda x: x == 'https://www.list.am/item/23006282?ld_src=2',hrefs): #hrefs:
    print(href)
    car_soup = get_specific_car_content(href)
    car_name = get_car_name(car_soup)
    price = get_car_price(car_soup)
    metadata = get_car_metadata(car_soup=car_soup)
    metadata = reverse_keys(metadata , new_keys=('Հնարավոր է փոխանակում','Մաքսազերծված է'))
    description,post_id,create_date,update_date = get_add_info(car_soup)
    seller_id = get_seller_id(car_soup)
    metadata['car_name']=car_name
    metadata['price'] = price
    metadata['description'] = description
    metadata['post_id'] = post_id
    metadata['create_date'] = create_date
    metadata['update_date'] = update_date
    metadata['Link'] = href
    cars_list.append(metadata)

https://www.list.am/item/23006282?ld_src=2


In [52]:
print(metadata)

{'Մակնիշ': 'Mercedes-Benz', 'Մոդել': 'CLS-Class AMG', 'Թափքի տեսակ': 'Սեդան', 'Տարի': '2012', 'Շարժիչ': 'Բենզին', 'Շարժիչի ծավալ': '3.0 L', 'Հզորություն': '306 ձ.ուժ', 'Փոխանցման տուփ': 'Ավտոմատ', 'Քարշակ': 'Ետևի', 'Վազք': '230,000 կմ', 'Ներկա վիճակը': 'Չվթարված', 'Գազի սարքավորումներ': 'Չտեղադրված', 'Ղեկ': 'Ձախ', 'Գույն': 'Բորդո', 'Անիվի չափսերը': 'R17', 'Լուսարձակներ': 'Քսենոն', 'Սրահի գույնը': 'Բեժ', 'Սրահը': 'Կաշի', 'Լյուկ': 'Սովորական', 'Օդորակիչ': 'Օդորակիչ', 'Տաքացվող նստատեղեր': 'Տաքացվող նստատեղեր', 'Տաքացվող ղեկ': 'Տաքացվող ղեկ', 'Օդափոխվող նստատեղեր': 'Օդափոխվող նստատեղեր', 'Կրուիզ-կոնտրոլ': 'Կրուիզ-կոնտրոլ', 'Մգեցված ապակիներ': 'Մգեցված ապակիներ', 'Հնարավոր է փոխանակում': 'Այո', 'car_name': 'Mercedes-Benz CLS-Class AMG, 3.0 լ, 2012 թ.', 'price': '$26,000', 'description': 'Գտնվում է լավ վիճակում հնարավորե պոխանակում', 'post_id': '23006282', 'create_date': '15.10.2025', 'update_date': '11.08.2026, 19:54', 'Link': 'https://www.list.am/item/23006282?ld_src=2'}


In [40]:
df = pd.DataFrame(cars_list)

In [41]:
df['scrape_date'] = pd.to_datetime(datetime.today())

In [42]:
df

,Մակնիշ,Մոդել,Թափքի տեսակ,Տարի,Շարժիչ,Շարժիչի ծավալ,Հզորություն,Փոխանցման տուփ,Քարշակ,Վազք,...,car_name,price,description,post_id,create_date,update_date,Link,Մաքսազերծված է,VIN,scrape_date
0,Mercedes-Benz,CLS-Class AMG,Սեդան,2012,Բենզին,3.0 L,306 ձ.ուժ,Ավտոմատ,Ետևի,"230,000 կմ",...,"Mercedes-Benz CLS-Class AMG, 3.0 լ, 2012 թ.","$26,000",Գտնվում է լավ վիճակում հնարավորե պոխանակում,23006282,15.10.2025,"11.08.2026, 19:54",https://www.list.am/item/23006282?ld_src=2,NaN,NaN,2026-08-11 23:26:18.287589
1,Mercedes-Benz,CLS-Class,Սեդան,2012,Բենզին,4.7 L,402 ձ.ուժ,Ավտոմատ,Ետևի,"270,000 կմ",...,"Mercedes-Benz CLS-Class, 4.7 լ, 2012 թ., շագան...","$22,000",CLS550 4.7 biturboՓՈԽԱՆԱԿՈՒՄ ԱՌԱՋԱՐԿԵՔՑանկալիա...,24078167,08.08.2026,"11.08.2026, 15:39",https://www.list.am/item/24078167?ld_src=2,NaN,NaN,2026-08-11 23:26:18.287589
2,Mercedes-Benz,CLS-Class AMG,Կուպե,2016,Բենզին,3.0 L,333 ձ.ուժ,Ավտոմատ,Ետևի,"110,000 կմ",...,"Mercedes-Benz CLS-Class AMG կուպե, 3.0 լ, 2016 թ.","$35,000",Պատրաստ ցանկացած ստուգման փոխանակում G Կլասի հետ,23587943,29.03.2026,"11.08.2026, 11:13",https://www.list.am/item/23587943?ld_src=2,NaN,NaN,2026-08-11 23:26:18.287589
3,Mercedes-Benz,CLS-Class AMG,Սեդան,2015,Բենզին,3.0 L,329 ձ.ուժ,Ավտոմատ,Լիաքարշակ,"142,000 կմ",...,"Mercedes-Benz CLS-Class AMG, 3.0 լ, լիաքարշ, 2...","$31,000","Վաճառվում է, ոչմի խնդիր չունի. Ռեալ մարդիկ զան...",24088412,11.08.2026,NaN,https://www.list.am/item/24088412?ld_src=2,Այո,NaN,2026-08-11 23:26:18.287589
4,Mercedes-Benz,CLS-Class,Կուպե,2014,Բենզին,4.7 L,402 ձ.ուժ,Ավտոմատ,Ետևի,"199,000 կմ",...,"Mercedes-Benz CLS-Class կուպե, 4.7 լ, 2014 թ., սև","$24,000","Վաճառվում է Mercedes Benz CLS 550 AMG Package,...",23921802,27.06.2026,"11.08.2026, 03:01",https://www.list.am/item/23921802?ld_src=2,NaN,NaN,2026-08-11 23:26:18.287589
5,Mercedes-Benz,CLS-Class,Կուպե,2012,Բենզին,4.7 L,408 ձ.ուժ,Ավտոմատ,Լիաքարշակ,"154,000 կմ",...,"Mercedes-Benz CLS-Class կուպե, 4.7 լ, լիաքարշ,...","$19,500",Mersedes cls գտնվում է շատ թարմ վիճակում w218 ...,23820362,29.05.2026,"11.08.2026, 00:29",https://www.list.am/item/23820362?ld_src=2,NaN,NaN,2026-08-11 23:26:18.287589
6,Mercedes-Benz,CLS-Class AMG,Կուպե,2012,Բենզին,3.0 L,306 ձ.ուժ,Ավտոմատ,Ետևի,"230,000 կմ",...,"Mercedes-Benz CLS-Class AMG կուպե, 3.0 լ, 2012 թ.","$26,000","3 գույն քամելեոն, սպասրկվել է միայն Մերսեդեսի ...",24029867,26.07.2026,"10.08.2026, 14:57",https://www.list.am/item/24029867?ld_src=2,NaN,NaN,2026-08-11 23:26:18.287589
7,Mercedes-Benz,CLS-Class,Սեդան,2016,Բենզին,3.0 L,333 ձ.ուժ,Ավտոմատ,Ետևի,"167,500 կմ",...,"Mercedes-Benz CLS-Class, 3.0 լ, 2016 թ.","$32,000",Մեքենան գտնվումա շատ լավ վիճակում չունի վոչմի ...,20848095,01.04.2024,"10.08.2026, 11:38",https://www.list.am/item/20848095?ld_src=2,NaN,NaN,2026-08-11 23:26:18.287589
8,Mercedes-Benz,CLS-Class,Սեդան,2016,Բենզին,3.0 L,333 ձ.ուժ,Ավտոմատ,Լիաքարշակ,"100,000 կմ",...,"Mercedes-Benz CLS-Class, 3.0 լ, լիաքարշ, 2016 թ.","$25,500",Շարժիչի ծավալ 3.0 բիտուրբոՓոխանակում հողի հետՄ...,20844810,31.03.2024,"10.08.2026, 11:17",https://www.list.am/item/20844810?ld_src=2,NaN,NaN,2026-08-11 23:26:18.287589
9,Mercedes-Benz,CLS-Class,Սեդան,2013,Բենզին,3.5 L,306 ձ.ուժ,Ավտոմատ,Ետևի,"167,000 կմ",...,"Mercedes-Benz CLS-Class, 3.5 լ, 2013 թ., սպիտակ","$25,500",Mercedes-Benz CLS 350 2013թ. (Japan)Արտաքնապես...,24084413,10.08.2026,NaN,https://www.list.am/item/24084413?ld_src=2,NaN,NaN,2026-08-11 23:26:18.287589


In [16]:


# 2. Define your PostgreSQL credentials
db_user = 'postgres'
db_password = 'postgres'
db_host = 'localhost'       # Use your server IP if it is not hosted locally
db_port = '5432'            # 5432 is the default PostgreSQL port
db_name = 'car_db'

# 3. Create the SQLAlchemy engine for PostgreSQL
connection_string = f'postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
engine = create_engine(connection_string)


In [17]:

# 4. Append the data to the table
df.to_sql(
    name='list_am_listings', 
    con=engine, 
    if_exists='append', 
    index=False
)

print(f"Successfully appended {len(df)} rows to PostgreSQL for {df['scrape_date'].iloc[0]}")

Successfully appended 27 rows to PostgreSQL for 2026-08-11 22:54:39.773282
